In [35]:
import os
from ensurepip import bootstrap
import warnings
from gc import set_debug, callbacks
from statistics import mean
from jupyter_builder.jupyterlab_semver import satisfies
from parso.python import pep8
from prompt_toolkit.key_binding.bindings.named_commands import yank_nth_arg
from sklearn.impute import SimpleImputer
from optuna.trial import FixedTrial
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from sklearn.inspection import permutation_importance
import shap
from optuna.visualization import plot_param_importances, plot_optimization_history, plot_slice
import math
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import TargetEncoder
from sklearn.base import clone

load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="adult")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM adults", engine)
df['workclass'] = df['workclass'].fillna('Private')
df['native_country'] = df['native_country'].fillna('United-States')
df = df.drop_duplicates()
top_6 = ['Prof-specialty',
    'Craft-repair',
    'Exec-managerial',
    'Adm-clerical',
    'Sales',
    'Other-service']
prods = df['occupation'].value_counts().loc[top_6]
prods = prods / prods.sum()
null_mask = df['occupation'].isna()
rng = np.random.default_rng(42)
df.loc[null_mask, 'occupation'] = rng.choice(top_6, size=null_mask.sum(), p=prods)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df.shape

(32537, 15)

In [36]:
x = df.drop(columns=['income',  'sex', 'education'])
y = df['income']

In [37]:
x_tmp, x_test, y_tmp, y_test  = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

In [38]:
t = ['workclass', 'marital_status', 'occupation', 'native_country', 'race', 'relationship']
prep = ColumnTransformer([('cat', TargetEncoder(cv=5, shuffle=True, random_state=42, smooth='auto'), t)], remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')

In [39]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_tr, y_tr)
print(accuracy_score(y_test, dummy.predict(x_test)))
print(roc_auc_score(y_test, dummy.predict_proba(x_test)[:, 1]))

0.7590657652120467
0.5


In [40]:
def unic(trial, model_name):
    if model_name == 'lr':
        params = {'C': trial.suggest_float('lr_C', 0.001, 1.0, log=True),
                  'penalty': trial.suggest_categorical('lr_penalty', ['l1', 'l2']),
                  'class_weight': trial.suggest_categorical('lr_class_weight', [None, 'balanced']),
                  'solver': 'liblinear',
                  'max_iter': 1000,
                  'random_state': 42}
        model = Pipeline([('scaler', StandardScaler()),
                          ('clf', LogisticRegression(**params))])
    elif model_name == 'rf':
        params = {'n_estimators': trial.suggest_int('rf_n_estimators', 100, 1000),
                  'max_depth': trial.suggest_int('rf_max_depth', 2, 13),
                  'min_saples_leaf': trial.suggest_int('rf_min_saples_leaf', 1, 30),
                  'max_features': trial.suggest_float('rf_max_features', 0.2, 1.0),
                  'min_samples_split': trial.suggest_int('rf_min_samples_split', 2, 30),
                  'random_state': 42,
                  'max_samples': trial.suggest_float('rf_max_samples', 0.4, 1.0),
                  'class_weight': trial.suggest_categorical('rf_class_weight', [None, 'balanced']),
                  'bootstrap': True,
                  'n_jobs': -1,}
        model = RandomForestClassifier(**params)
    elif model_name == 'xgb':
        params = {'n_estimators': 1000,
                  'max_depth': trial.suggest_int('xgb_max_depth', 2, 10),
                  'learning_rate': trial.suggest_float('xgb_learning_rate', 0.005, 1.0, log=True),
                  'subsample': trial.suggest_float('xgb_subsample', 0.65, 1.0),
                  'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.6, 1.0),
                  'reg_alpha': trial.suggest_float('xgb_reg_alpha', 0.001, 4.0, log=True),
                  'reg_lambda': trial.suggest_float('xgb_reg_lambda', 0.001, 4.0, log=True),
                  'eval_metric': ['error', 'auc'],
                  'early_stopping_rounds': 50,
                  'tree_method': 'hist',
                  'random_state': 42,
                  'n_jobs': -1,
                  'max_bin': trial.suggest_int('xgb_max_bins', 64, 512),
                  'min_child_weight': trial.suggest_int('xgb_min_child_weight', 1, 30, log=True),
                  'objective': 'binary:logistic',}
        model = XGBClassifier(**params)
    elif model_name == 'lgb':
        params = {'n_estimators': 1000,
                  'learning_rate': trial.suggest_float('lgb_learning_rate', 0.005, 1.0, log=True),
                  'num_leaves': trial.suggest_int('lgb_num_leaves', 8, 256, log=True),
                  'min_child_samples': trial.suggest_int('lgb_min_child_samples', 8, 200, log=True),
                  'subsample': trial.suggest_float('lgb_subsample', 0.6, 1.0),
                  'subsample_freq': 1,
                  'colsample_bytree': trial.suggest_float('lgb_colsample_bytee', 0.6, 1.0),
                  'reg_alpha': trial.suggest_float('lgb_reg_alpha', 0.001, 4.0, log=True),
                  'reg_lambda': trial.suggest_float('lgb_reg_lambda', 0.001, 4.0, log=True),
                  'n_jobs': -1,
                  'metric': ['auc', 'binary_logloss'],
                  'random_state': 42,
                  'verbose':False,
                  'max_bin': trial.suggest_int('lgb_max_bins', 64, 512),
                  'path_smooth': trial.suggest_int('lgb_path_smooth', 0.0, 10.0),}
        model = LGBMClassifier(**params)
    elif model_name == 'cat':
        params = {'n_estimators': 1000,
                  'learning_rate': trial.suggest_float('cat_learning_rate', 0.005, 1.0, log=True),
                  'depth': trial.suggest_int('cat_depth', 2, 10),
                  'l2_leaf_reg': trial.suggest_float('cat_l2_leaf_reg', 0.01, 10.0, log=True),
                  'random_state': 42,
                  'random_strength': trial.suggest_float('cat_random_strength', 0.001, 10.0, log=True),
                  'subsample': trial.suggest_float('cat_subsample', 0.6, 1.0),
                  'bootstrap_type': 'MVS',
                  'n_jobs': -1,
                  'eval_metric': 'AUC',
                  'loss_function': 'Logloss',
                  'custom_metric': ['AUC', 'Accuracy'],
                  'verbosity': -1,
                  'early_stopping_rounds': 50,
                  'border_count': trial.suggest_int('cat_border_count', 32, 152),
                  'auto_class_weights': 'Balanced',}
        model = CatBoostClassifier(**params)
    return model

In [41]:
def fit_model(model, name, x_tr, y_tr, x_val, y_val):
    if name == 'xgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False, early_stopping_rounds=50)
        return model.best_iteration + 1
    if name == 'lgb':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False), lgb.log_evaluation(0)],)
        return model.best_iteration_
    if name == 'cat':
        model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False, early_stopping_rounds=50)
        return model.get_best_iteration() + 1
    model.fit(x_tr, y_tr)
    return None

In [42]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
def make_objective(name):
    def objective(trial):
        scores, iters = [], []
        for tr_idx, val_idx in skf.split(x_tmp, y_tmp):
            xa, xb = x_tmp.iloc[tr_idx], x_tmp.iloc[val_idx]
            ya, yb = y_tmp.iloc[tr_idx], y_tmp.iloc[val_idx]

            p = clone(prep)
            xa_t = p.fit_transform(xa, ya)
            xb_t = p.transform(xb)

            model = unic(trial, name)
            n_trees = fit_model(model, name, xa_t, ya, xb_t, yb)
            proba = model.predict_proba(xb_t)[:, 1]
            scores.append(roc_auc_score(yb, proba))
            if n_trees is not None:
                iters.append(n_trees)
        if iters:
            trial.set_user_attr("best_iter", int(np.mean(iters)))
        trial.set_user_attr("auc_std", float(np.std(scores)))
        return float(np.mean(scores))
    return objective


In [43]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

n_trials = {'lr': 15, 'cat': 15, 'xgb': 18, 'lgb': 18, 'rf': 13}
studies = {}
for name, n in n_trials.items():
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=5),
        study_name=name,
    )
    study.optimize(make_objective(name), n_trials=n)
    studies[name] = study
    t = study.best_trial
    print(f'{name}: auc = {study.best_value:.5f} +- {t.user_attrs['auc_std']:.5f}')

/home/artem/Adult_ds/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/home/artem/Adult_ds/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/artem/Adult_ds/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. pena

lr: auc = 0.90119 +- 0.00186


TypeError: CatBoostClassifier.__init__() got an unexpected keyword argument 'n_jobs'